<a href="https://colab.research.google.com/github/RachmadAprisandhy/PBL_Kelompok3/blob/Zacky/Klasifikasi_Grading_Pisang_CNN_Gabungan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Langkah 1 — Hubungkan Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

print("Google Drive berhasil dihubungkan.")

Mounted at /content/drive
Google Drive berhasil dihubungkan.


Langkah 2 — Tentukan folder Gabungan Dataset

In [ ]:
import os

folder_dataset = (
    "/content/drive/MyDrive/Dataset PBL/"
    "Banana Images/Gabungan Dataset"
)

print("Folder ditemukan:", os.path.isdir(folder_dataset))

print("\nIsi folder:")
if os.path.isdir(folder_dataset):
    print(sorted(os.listdir(folder_dataset)))

Folder ditemukan: True

Isi folder:
['test', 'train']


Langkah 3 — Periksa jumlah gambar setiap kelas

In [ ]:
import os

folder_dataset = (
    "/content/drive/MyDrive/Dataset PBL/"
    "Banana Images/Gabungan Dataset"
)

kelas = ["Grade A", "Grade B", "Grade C", "Grade D"]
ekstensi = (".png", ".jpg", ".jpeg", ".bmp", ".webp")

total_semua = 0

for bagian in ["train", "test"]:
    folder_bagian = os.path.join(folder_dataset, bagian)
    total_bagian = 0

    print(f"\n===== {bagian.upper()} =====")

    for nama_kelas in kelas:
        folder_kelas = os.path.join(folder_bagian, nama_kelas)

        if not os.path.isdir(folder_kelas):
            print(f"{nama_kelas}: folder tidak ditemukan")
            continue

        jumlah = sum(
            1
            for nama in os.listdir(folder_kelas)
            if nama.lower().endswith(ekstensi)
        )

        total_bagian += jumlah
        print(f"{nama_kelas}: {jumlah} gambar")

    print(f"Total {bagian}: {total_bagian} gambar")
    total_semua += total_bagian

print("\n==============================")
print("TOTAL SELURUH DATA:", total_semua)
print("==============================")



===== TRAIN =====
Grade A: 2001 gambar
Grade B: 2001 gambar
Grade C: 2000 gambar
Grade D: 2000 gambar
Total train: 8002 gambar

===== TEST =====
Grade A: 500 gambar
Grade B: 500 gambar
Grade C: 500 gambar
Grade D: 500 gambar
Total test: 2000 gambar

TOTAL SELURUH DATA: 10002


Langkah 4 — Preprocessing dan Memuat Dataset (224 × 224)

In [ ]:
import tensorflow as tf
import os

folder_dataset = (
    "/content/drive/MyDrive/Dataset PBL/"
    "Banana Images/Gabungan Dataset"
)

ukuran_gambar = (224, 224)
batch_size = 16
seed = 42

# Memuat data training
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(folder_dataset, "train"),
    labels="inferred",
    label_mode="int",
    image_size=ukuran_gambar,
    batch_size=batch_size,
    shuffle=True,
    seed=seed
)

# Memuat data testing
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(folder_dataset, "test"),
    labels="inferred",
    label_mode="int",
    image_size=ukuran_gambar,
    batch_size=batch_size,
    shuffle=False
)

# Menyimpan urutan kelas
nama_kelas = train_ds.class_names

print("\nUrutan kelas:", nama_kelas)
print("Jumlah kelas:", len(nama_kelas))

# Normalisasi piksel menjadi rentang 0–1
train_ds = train_ds.map(
    lambda gambar, label: (
        tf.cast(gambar, tf.float32) / 255.0,
        label
    )
)

test_ds = test_ds.map(
    lambda gambar, label: (
        tf.cast(gambar, tf.float32) / 255.0,
        label
    )
)

# Mengoptimalkan proses input data
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("\nPreprocessing selesai.")


Found 8002 files belonging to 4 classes.
Found 2000 files belonging to 4 classes.

Urutan kelas: ['Grade A', 'Grade B', 'Grade C', 'Grade D']
Jumlah kelas: 4

Preprocessing selesai.


Langkah 5 — Memeriksa Bentuk dan Ukuran Dataset

In [ ]:
# Mengambil satu batch dari dataset training dan testing
for gambar, label in train_ds.take(1):
    print("===== DATA TRAINING =====")
    print("Bentuk gambar:", gambar.shape)
    print("Bentuk label :", label.shape)
    print("Tipe data    :", gambar.dtype)
    print("Nilai piksel :", float(tf.reduce_min(gambar)),
          "sampai", float(tf.reduce_max(gambar)))

for gambar, label in test_ds.take(1):
    print("\n===== DATA TESTING =====")
    print("Bentuk gambar:", gambar.shape)
    print("Bentuk label :", label.shape)

print("\nUrutan kelas:", nama_kelas)
print("Jumlah kelas:", len(nama_kelas))

===== DATA TRAINING =====
Bentuk gambar: (16, 224, 224, 3)
Bentuk label : (16,)
Tipe data    : <dtype: 'float32'>
Nilai piksel : 0.0 sampai 1.0

===== DATA TESTING =====
Bentuk gambar: (16, 224, 224, 3)
Bentuk label : (16,)

Urutan kelas: ['Grade A', 'Grade B', 'Grade C', 'Grade D']
Jumlah kelas: 4


Langkah 6 — Membangun Arsitektur CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Membuat arsitektur CNN baru
model = models.Sequential([
    # Input gambar 224 x 224 x 3 (RGB)
    layers.Input(shape=(224, 224, 3)),

    # Blok CNN 1
    layers.Conv2D(
        32, (3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Blok CNN 2
    layers.Conv2D(
        64, (3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Blok CNN 3
    layers.Conv2D(
        128, (3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Merangkum fitur gambar
    layers.GlobalAveragePooling2D(),

    # Klasifikasi
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(4, activation="softmax")
])

# Menampilkan arsitektur model
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,764 (397.52 KB)

 Trainable params: 101,764 (397.52 KB)

 Non-trainable params: 0 (0.00 B)

Langkah 7 — Compile Model CNN

In [ ]:
# Compile model CNN
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model berhasil di-compile.")
print("Optimizer : Adam")
print("Loss      : Sparse Categorical Crossentropy")
print("Metric    : Accuracy")

Model berhasil di-compile.
Optimizer : Adam
Loss      : Sparse Categorical Crossentropy
Metric    : Accuracy


Langkah 8 — Training Model CNN

In [ ]:
# Menentukan jumlah epoch
jumlah_epoch = 20

# Melatih model CNN
history = model.fit(
    train_ds,
    epochs=jumlah_epoch,
    verbose=1
)

print("\nTraining model CNN selesai.")

Epoch 1/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 1022s 2s/step - accuracy: 0.9308 - loss: 0.1826
Epoch 2/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 1007s 2s/step - accuracy: 0.9370 - loss: 0.1494
Epoch 3/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 997s 2s/step - accuracy: 0.9443 - loss: 0.1351
Epoch 4/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 983s 2s/step - accuracy: 0.9476 - loss: 0.1330
Epoch 5/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 1054s 2s/step - accuracy: 0.9446 - loss: 0.1270
Epoch 6/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 1002s 2s/step - accuracy: 0.9451 - loss: 0.1205
Epoch 7/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 1025s 2s/step - accuracy: 0.9493 - loss: 0.1192
Epoch 8/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 982s 2s/step - accuracy: 0.9526 - loss: 0.1107
Epoch 9/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 1043s 2s/step - accuracy: 0.9564 - loss: 0.1065
Epoch 10/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 1046s 2s/step - accuracy: 0.9514 - loss: 0.1119
Epoch 11/20
501/501 ━━━━━━━━━━━━━━━━━━━━ 1029s 2s/step - accuracy: 0.9541 - loss: 0.1038
Epoch 12/20
501/501 ━━━━━━━━━━━━━

Langkah 9 — Evaluasi Model CNN

In [ ]:
# Evaluasi model menggunakan data testing
test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print("\n===== HASIL EVALUASI MODEL CNN =====")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")

Langkah 10 — Evaluasi Detail Model CNN

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report

# 1. Mengambil label sebenarnya dari data testing
y_true = np.concatenate([
    label.numpy()
    for gambar, label in test_ds
])

# 2. Melakukan prediksi menggunakan model CNN
y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

# 3. Membuat confusion matrix
label_kelas = np.arange(len(nama_kelas))

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=label_kelas
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=nama_kelas,
    yticklabels=nama_kelas
)

plt.title("Confusion Matrix - CNN Pisang")
plt.xlabel("Prediksi Model")
plt.ylabel("Label Sebenarnya")
plt.tight_layout()
plt.show()

# 4. Menghitung prediksi benar, salah, dan akurasi setiap Grade
hasil_grade = []

for i, grade in enumerate(nama_kelas):
    benar = int(cm[i, i])
    total_grade = int(np.sum(cm[i, :]))
    salah = total_grade - benar

    akurasi_grade = (
        benar / total_grade * 100
        if total_grade > 0 else 0
    )

    hasil_grade.append({
        "Grade": grade,
        "Jumlah Data": total_grade,
        "Prediksi Benar": benar,
        "Prediksi Salah": salah,
        "Akurasi (%)": round(akurasi_grade, 2)
    })

df_grade = pd.DataFrame(hasil_grade)

print("\n===== JUMLAH PREDIKSI BENAR DAN SALAH PER GRADE =====")
print(df_grade.to_string(index=False))

# 5. Menghitung akurasi keseluruhan model
total_data = int(np.sum(cm))
total_benar = int(np.trace(cm))
total_salah = total_data - total_benar

akurasi_model = (
    total_benar / total_data * 100
    if total_data > 0 else 0
)

print("\n===== AKURASI KESELURUHAN MODEL CNN =====")
print("Total data testing :", total_data)
print("Prediksi benar     :", total_benar)
print("Prediksi salah     :", total_salah)
print(f"Akurasi model      : {akurasi_model:.2f}%")

# 6. Menampilkan classification report
print("\n===== CLASSIFICATION REPORT =====")
print(
    classification_report(
        y_true,
        y_pred,
        labels=label_kelas,
        target_names=nama_kelas,
        zero_division=0,
        digits=4
    )
)

# 7. Grafik jumlah prediksi benar dan salah per Grade
df_grade.plot(
    x="Grade",
    y=["Prediksi Benar", "Prediksi Salah"],
    kind="bar",
    figsize=(9, 5)
)

plt.title("Prediksi Benar dan Salah per Grade")
plt.xlabel("Grade Pisang")
plt.ylabel("Jumlah Gambar")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 8. Grafik akurasi setiap Grade
plt.figure(figsize=(8, 5))
bars = plt.bar(
    df_grade["Grade"],
    df_grade["Akurasi (%)"]
)

plt.title("Akurasi Model CNN pada Setiap Grade")
plt.xlabel("Grade Pisang")
plt.ylabel("Akurasi (%)")
plt.ylim(0, 100)

for bar, nilai in zip(bars, df_grade["Akurasi (%)"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{nilai:.2f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

Langkah 11 — Visualisasi Training Model CNN

In [ ]:
import matplotlib.pyplot as plt

# Mengambil riwayat akurasi dan loss
akurasi = history.history["accuracy"]
loss = history.history["loss"]
epoch = range(1, len(akurasi) + 1)

# Membuat grafik
plt.figure(figsize=(12, 5))

# Grafik akurasi training
plt.subplot(1, 2, 1)
plt.plot(epoch, akurasi, marker="o")
plt.title("Akurasi Training Model CNN")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.xticks(list(epoch))
plt.grid(True)

# Grafik loss training
plt.subplot(1, 2, 2)
plt.plot(epoch, loss, marker="o")
plt.title("Loss Training Model CNN")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(list(epoch))
plt.grid(True)

plt.tight_layout()
plt.show()